In [2]:
# skin_app.py
import os
import sys
import numpy as np
import cv2
import tensorflow as tf
from PyQt5.QtCore import Qt, QThread, pyqtSignal
from PyQt5.QtGui import QPixmap, QImage, QFont, QColor
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QPushButton, QLabel, QFileDialog, QListWidget, QListWidgetItem,
    QMessageBox, QGroupBox, QSplitter, QSizePolicy
)
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure

from model_builder import (
    load_model_from_weights, predict_image, extract_handcrafted_features_bgr,
    CLASS_NAMES, NUM_CLASSES, HANDCRAFT_DIM, IMG_SIZE
)
from tensorflow.keras.applications.efficientnet import preprocess_input as ef_preprocess


# ======================= دوال Grad-CAM المتقدمة =======================
def get_gradcam_heatmap(model, img_array, hand_features=None, pred_index=None, layer_name='convnext_tiny_stage_3_block_2_pointwise_conv_2'):
    grad_model = None
    try:
        grad_model = tf.keras.models.Model(
            model.inputs,
            [model.get_layer(layer_name).output, model.output]
        )
    except Exception:
        for layer in reversed(model.layers):
            if 'conv' in layer.name and 'pointwise' in layer.name:
                grad_model = tf.keras.models.Model(
                    model.inputs, [layer.output, model.output]
                )
                break

    if grad_model is None:
        raise ValueError(f"Could not find layer '{layer_name}' or an alternative.")

    inputs = [img_array]
    if hand_features is not None:
        inputs.append(hand_features)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


# ======================= Thread للتحميل =======================
class ModelLoaderThread(QThread):
    finished = pyqtSignal(object)
    error = pyqtSignal(str)

    def __init__(self, weights_path):
        super().__init__()
        self.weights_path = weights_path

    def run(self):
        try:
            model = load_model_from_weights(self.weights_path)
            self.finished.emit(model)
        except Exception as e:
            self.error.emit(str(e))


# ======================= Thread للتنبؤ واستخراج Grad-CAM =======================
class PredictThread(QThread):
    finished = pyqtSignal(object, np.ndarray, np.ndarray)
    error = pyqtSignal(str)

    def __init__(self, model, img_rgb, image_path):
        super().__init__()
        self.model = model
        self.img_rgb = img_rgb
        self.image_path = image_path

    def run(self):
        try:
            # 1. التنبؤ
            probs = predict_image(self.model, self.img_rgb)
            
            # 2. توليد الخريطة الحرارية Grad-CAM
            img_resized = cv2.resize(self.img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
            img_pre = ef_preprocess(img_resized)[np.newaxis, ...]
            
            img_bgr = cv2.cvtColor(self.img_rgb, cv2.COLOR_RGB2BGR)
            hand_feat = extract_handcrafted_features_bgr(img_bgr)[np.newaxis, ...]
            
            heatmap = get_gradcam_heatmap(
                self.model, img_pre, hand_feat,
                pred_index=int(np.argmax(probs))
            )
            
            self.finished.emit(probs, heatmap, img_bgr)
        except Exception as e:
            self.error.emit(str(e))


# ======================= النافذة الرئيسية =======================
class SkinClassifierApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Skin Lesion Classifier & Grad-CAM — ConvNeXtTiny")
        self.resize(1400, 780)
        self.model = None
        self.current_img_rgb = None
        self.current_img_path = None
        self._setup_style()
        self._build_ui()

    def _setup_style(self):
        self.setStyleSheet("""
            QMainWindow { background-color: #1e1e2e; }
            QWidget { color: #e0e0e0; font-family: 'Segoe UI'; font-size: 13px; }
            QGroupBox {
                background-color: #2a2a3d;
                border: 1px solid #3a3a55;
                border-radius: 8px;
                margin-top: 12px;
                padding-top: 10px;
                font-weight: bold;
            }
            QGroupBox::title {
                subcontrol-origin: margin;
                left: 12px;
                padding: 0 6px;
                color: #89b4fa;
            }
            QPushButton {
                background-color: #45475a;
                border: none;
                border-radius: 6px;
                padding: 9px 16px;
                font-weight: bold;
            }
            QPushButton:hover { background-color: #585b70; }
            QPushButton:pressed { background-color: #313244; }
            QPushButton:disabled { background-color: #2a2a3d; color: #6c6c80; }
            QPushButton#primaryBtn { background-color: #89b4fa; color: #1e1e2e; }
            QPushButton#primaryBtn:hover { background-color: #a6c8ff; }
            QLabel#imageLabel {
                background-color: #11111b;
                border: 2px dashed #45475a;
                border-radius: 8px;
            }
            QLabel#statusLabel { color: #a6adc8; }
            QListWidget {
                background-color: #11111b;
                border: 1px solid #3a3a55;
                border-radius: 6px;
                padding: 4px;
            }
        """)

    def _build_ui(self):
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout(central)
        main_layout.setContentsMargins(14, 14, 14, 14)
        main_layout.setSpacing(10)

        # شريط التحميل العلوي
        top_bar = QHBoxLayout()
        self.load_btn = QPushButton("📂 تحميل ملف الأوزان (model.weights.h5)")
        self.load_btn.clicked.connect(self.on_load_weights)
        self.load_btn.setObjectName("primaryBtn")

        self.model_status = QLabel("⚠️ لم يتم تحميل أي موديل بعد")
        self.model_status.setObjectName("statusLabel")

        top_bar.addWidget(self.load_btn)
        top_bar.addWidget(self.model_status, 1)
        main_layout.addLayout(top_bar)

        # تقسيم الشاشة الرئيسي
        splitter = QSplitter(Qt.Horizontal)

        # قسم الصور (Original vs Heatmap vs Overlay)
        img_group = QGroupBox("🖼️ التحليل البصري (Original vs Heatmap vs Overlay)")
        img_layout = QVBoxLayout(img_group)

        images_layout = QHBoxLayout()
        
        # 1. الصورة الأصلية
        self.image_label = QLabel("Original Image")
        self.image_label.setAlignment(Qt.AlignCenter)
        self.image_label.setObjectName("imageLabel")
        self.image_label.setMinimumSize(280, 280)
        self.image_label.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)

        # 2. الخريطة الحرارية وحدها
        self.heatmap_label = QLabel("Heatmap Only")
        self.heatmap_label.setAlignment(Qt.AlignCenter)
        self.heatmap_label.setObjectName("imageLabel")
        self.heatmap_label.setMinimumSize(280, 280)
        self.heatmap_label.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)

        # 3. الدمج التراكبي (Overlay)
        self.overlay_label = QLabel("Grad-CAM Overlay")
        self.overlay_label.setAlignment(Qt.AlignCenter)
        self.overlay_label.setObjectName("imageLabel")
        self.overlay_label.setMinimumSize(280, 280)
        self.overlay_label.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)

        images_layout.addWidget(self.image_label)
        images_layout.addWidget(self.heatmap_label)
        images_layout.addWidget(self.overlay_label)
        img_layout.addLayout(images_layout, 1)

        btn_row = QHBoxLayout()
        self.open_btn = QPushButton("📷 فتح صورة")
        self.open_btn.clicked.connect(self.on_open_image)
        self.predict_btn = QPushButton("🔍 تنبؤ وتحليل")
        self.predict_btn.clicked.connect(self.on_predict)
        self.predict_btn.setObjectName("primaryBtn")
        self.predict_btn.setEnabled(False)
        self.clear_btn = QPushButton("🗑️ مسح")
        self.clear_btn.clicked.connect(self.on_clear)

        btn_row.addWidget(self.open_btn)
        btn_row.addWidget(self.predict_btn)
        btn_row.addWidget(self.clear_btn)
        img_layout.addLayout(btn_row)

        splitter.addWidget(img_group)

        # قسم النتائج والرسوم البيانية
        res_group = QGroupBox("📊 النتائج والاحتمالات")
        res_layout = QVBoxLayout(res_group)

        self.top_label = QLabel("—")
        self.top_label.setAlignment(Qt.AlignCenter)
        self.top_label.setStyleSheet(
            "font-size: 20px; font-weight: bold; color: #a6e3a1; padding: 6px;"
        )
        res_layout.addWidget(self.top_label)

        self.figure = Figure(figsize=(4, 3), dpi=90, facecolor="#2a2a3d")
        self.canvas = FigureCanvas(self.figure)
        self.canvas.setStyleSheet("background-color: #2a2a3d;")
        res_layout.addWidget(self.canvas, 1)

        self.prob_list = QListWidget()
        self.prob_list.setMinimumHeight(140)
        res_layout.addWidget(QLabel("جميع الاحتمالات:"))
        res_layout.addWidget(self.prob_list)

        splitter.addWidget(res_group)
        splitter.setSizes([750, 450])
        main_layout.addWidget(splitter, 1)

        self.status_bar = QLabel("جاهز")
        self.status_bar.setObjectName("statusLabel")
        main_layout.addWidget(self.status_bar)

    def on_load_weights(self):
        path, _ = QFileDialog.getOpenFileName(
            self, "اختر ملف الأوزان", "",
            "Keras Weights (*.h5 *.keras *.weights.h5);;All Files (*)"
        )
        if not path:
            return
        self.load_btn.setEnabled(False)
        self.model_status.setText(f"⏳ جاري تحميل: {os.path.basename(path)} ...")
        self.status_bar.setText("جاري بناء الموديل وتحميل الأوزان...")

        self.loader = ModelLoaderThread(path)
        self.loader.finished.connect(self._on_model_loaded)
        self.loader.error.connect(self._on_model_error)
        self.loader.start()

    def _on_model_loaded(self, model):
        self.model = model
        self.load_btn.setEnabled(True)
        self.model_status.setText("✅ تم تحميل الموديل بنجاح")
        self.status_bar.setText("الموديل جاهز — افتح صورة واضغط تنبؤ")
        self._update_predict_btn()
        QMessageBox.information(self, "نجاح", "تم تحميل الموديل بنجاح ✅")

    def _on_model_error(self, err):
        self.load_btn.setEnabled(True)
        self.model_status.setText("❌ فشل تحميل الموديل")
        self.status_bar.setText("خطأ في تحميل الموديل")
        QMessageBox.critical(self, "خطأ في تحميل الموديل", err)

    def on_open_image(self):
        path, _ = QFileDialog.getOpenFileName(
            self, "اختر صورة", "", "Images (*.png *.jpg *.jpeg *.bmp)"
        )
        if path:
            self.load_image(path)

    def load_image(self, path):
        try:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                raise ValueError("تعذر قراءة الصورة")
            self.current_img_path = path
            self.current_img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            self._show_images(self.current_img_rgb, None)
            self.status_bar.setText(f"تم تحميل: {os.path.basename(path)}")
            self._update_predict_btn()
            self._clear_results()
        except Exception as e:
            QMessageBox.warning(self, "خطأ", str(e))

    def _show_images(self, img_rgb, heatmap):
        h, w, ch = img_rgb.shape
        
        # 1. عرض الصورة الأصلية
        qimg = QImage(img_rgb.data, w, h, ch * w, QImage.Format_RGB888)
        pix = QPixmap.fromImage(qimg)
        self.image_label.setPixmap(
            pix.scaled(self.image_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation)
        )

        if heatmap is not None:
            heatmap_resized = cv2.resize(heatmap, (w, h))
            
            # 2. الخريطة الحرارية وحدها (Heatmap Only)
            heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
            heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
            qimg_heat = QImage(heatmap_rgb.data, w, h, ch * w, QImage.Format_RGB888)
            pix_heat = QPixmap.fromImage(qimg_heat)
            self.heatmap_label.setPixmap(
                pix_heat.scaled(self.heatmap_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation)
            )

            # 3. الدمج التراكبي (Grad-CAM Overlay)
            overlay = cv2.addWeighted(img_rgb, 0.6, heatmap_rgb, 0.4, 0)
            qimg_cam = QImage(overlay.data, w, h, ch * w, QImage.Format_RGB888)
            pix_cam = QPixmap.fromImage(qimg_cam)
            self.overlay_label.setPixmap(
                pix_cam.scaled(self.overlay_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation)
            )
        else:
            self.heatmap_label.setText("Heatmap Only")
            self.overlay_label.setText("Grad-CAM Overlay")

    def _update_predict_btn(self):
        self.predict_btn.setEnabled(self.model is not None and self.current_img_rgb is not None)

    def on_predict(self):
        if self.model is None or self.current_img_rgb is None:
            return
        self.predict_btn.setEnabled(False)
        self.status_bar.setText("⏳ جاري التنبؤ وحساب Grad-CAM...")
        
        self.predict_thread = PredictThread(self.model, self.current_img_rgb, self.current_img_path)
        self.predict_thread.finished.connect(self._on_predicted)
        self.predict_thread.error.connect(self._on_predict_error)
        self.predict_thread.start()

    def _on_predicted(self, probs, heatmap, img_bgr):
        self._update_results(probs)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        self._show_images(img_rgb, heatmap)
        
        self.predict_btn.setEnabled(True)
        top_idx = int(np.argmax(probs))
        self.status_bar.setText(f"✅ التنبؤ: {CLASS_NAMES[top_idx]} ({probs[top_idx]*100:.2f}%)")

    def _on_predict_error(self, err):
        self.predict_btn.setEnabled(True)
        self.status_bar.setText("خطأ في التنبؤ")
        QMessageBox.critical(self, "خطأ", err)

    def _clear_results(self):
        self.top_label.setText("—")
        self.prob_list.clear()
        self.figure.clear()
        self.canvas.draw()
        self.heatmap_label.setText("Heatmap Only")
        self.overlay_label.setText("Grad-CAM Overlay")

    def _update_results(self, probs):
        top_idx = int(np.argmax(probs))
        self.top_label.setText(f"🏆 {CLASS_NAMES[top_idx]}\n{probs[top_idx]*100:.2f}%")

        self.prob_list.clear()
        sorted_idx = np.argsort(probs)[::-1]
        for i in sorted_idx:
            name = CLASS_NAMES[i]
            pct = probs[i] * 100
            item = QListWidgetItem(f"{name:>8s}    {pct:6.2f}%")
            item.setForeground(QColor("#89b4fa") if i == top_idx else QColor("#cdd6f4"))
            self.prob_list.addItem(item)

        self.figure.clear()
        ax = self.figure.add_subplot(111)
        ax.set_facecolor("#2a2a3d")
        colors = ["#89b4fa" if i == top_idx else "#585b70" for i in range(len(probs))]
        bars = ax.bar(CLASS_NAMES, probs * 100, color=colors, edgecolor="#11111b")
        ax.set_ylim(0, 100)
        ax.set_ylabel("الاحتمال (%)", color="#cdd6f4")
        ax.tick_params(colors="#cdd6f4", labelsize=8)
        for s in ax.spines.values():
            s.set_color("#45475a")
        ax.grid(axis="y", alpha=0.2, color="#cdd6f4")
        self.figure.tight_layout()
        self.canvas.draw()

    def on_clear(self):
        self.current_img_rgb = None
        self.current_img_path = None
        self.image_label.setText("Original Image")
        self.image_label.setPixmap(QPixmap())
        self._clear_results()
        self._update_predict_btn()
        self.status_bar.setText("تم المسح")


def main():
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    win = SkinClassifierApp()
    win.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()

SystemExit: 0

In [3]:
# skin_app.py
import os
import sys
import numpy as np
import cv2
import tensorflow as tf
from PyQt5.QtCore import Qt, QThread, pyqtSignal, QSize
from PyQt5.QtGui import (
    QPixmap, QImage, QFont, QColor, QPainter, QPen, QBrush
)
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QPushButton, QLabel, QFileDialog, QListWidget, QListWidgetItem,
    QMessageBox, QGroupBox, QSplitter, QSizePolicy, QFrame,
    QProgressBar, QGraphicsDropShadowEffect
)
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure

from model_builder import (
    load_model_from_weights, predict_image, extract_handcrafted_features_bgr,
    CLASS_NAMES, NUM_CLASSES, HANDCRAFT_DIM, IMG_SIZE
)
from tensorflow.keras.applications.efficientnet import preprocess_input as ef_preprocess


# ======================= Palette =======================
PALETTE = {
    "base":      "#1e1e2e",
    "mantle":    "#181825",
    "crust":     "#11111b",
    "surface0":  "#313244",
    "surface1":  "#45475a",
    "surface2":  "#585b70",
    "overlay":   "#6c7086",
    "subtext":   "#a6adc8",
    "text":      "#cdd6f4",
    "blue":      "#89b4fa",
    "sky":       "#89dceb",
    "green":     "#a6e3a1",
    "yellow":    "#f9e2af",
    "red":       "#f38ba8",
    "mauve":     "#cba6f7",
    "teal":      "#94e2d5",
}


# ======================= Grad-CAM =======================
def get_gradcam_heatmap(model, img_array, hand_features=None,
                        pred_index=None,
                        layer_name='convnext_tiny_stage_3_block_2_pointwise_conv_2'):
    grad_model = None
    try:
        grad_model = tf.keras.models.Model(
            model.inputs,
            [model.get_layer(layer_name).output, model.output]
        )
    except Exception:
        for layer in reversed(model.layers):
            if 'conv' in layer.name and 'pointwise' in layer.name:
                grad_model = tf.keras.models.Model(
                    model.inputs, [layer.output, model.output]
                )
                break

    if grad_model is None:
        raise ValueError(f"Could not find layer '{layer_name}' or an alternative.")

    inputs = [img_array]
    if hand_features is not None:
        inputs.append(hand_features)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


# ======================= Threads =======================
class ModelLoaderThread(QThread):
    finished = pyqtSignal(object)
    error    = pyqtSignal(str)

    def __init__(self, weights_path):
        super().__init__()
        self.weights_path = weights_path

    def run(self):
        try:
            model = load_model_from_weights(self.weights_path)
            self.finished.emit(model)
        except Exception as e:
            self.error.emit(str(e))


class PredictThread(QThread):
    finished = pyqtSignal(object, np.ndarray, np.ndarray)
    error    = pyqtSignal(str)

    def __init__(self, model, img_rgb, image_path):
        super().__init__()
        self.model      = model
        self.img_rgb    = img_rgb
        self.image_path = image_path

    def run(self):
        try:
            probs = predict_image(self.model, self.img_rgb)

            img_resized = cv2.resize(self.img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
            img_pre     = ef_preprocess(img_resized)[np.newaxis, ...]

            img_bgr  = cv2.cvtColor(self.img_rgb, cv2.COLOR_RGB2BGR)
            hand_feat = extract_handcrafted_features_bgr(img_bgr)[np.newaxis, ...]

            heatmap = get_gradcam_heatmap(
                self.model, img_pre, hand_feat,
                pred_index=int(np.argmax(probs))
            )

            self.finished.emit(probs, heatmap, img_bgr)
        except Exception as e:
            self.error.emit(str(e))


# ======================= Image Drop Zone =======================
class ImageDropLabel(QLabel):
    """QLabel that accepts image drag-and-drop."""
    fileDropped = pyqtSignal(str)

    def __init__(self, text=""):
        super().__init__(text)
        self.setAcceptDrops(True)

    def dragEnterEvent(self, event):
        if event.mimeData().hasUrls():
            event.acceptProposedAction()

    def dropEvent(self, event):
        for url in event.mimeData().urls():
            path = url.toLocalFile()
            if os.path.splitext(path)[1].lower() in (".png", ".jpg", ".jpeg", ".bmp"):
                self.fileDropped.emit(path)
                break
        event.acceptProposedAction()


# ======================= Main Window =======================
class SkinClassifierApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("DermaVision — Skin Lesion Classifier with Grad-CAM")
        self.resize(1500, 840)
        self.setMinimumSize(1200, 700)

        self.model             = None
        self.current_img_rgb   = None
        self.current_img_path  = None
        self._loader           = None
        self._predictor        = None

        self._setup_style()
        self._build_ui()
        self._update_predict_btn()

    # ---------- Style ----------
    def _setup_style(self):
        self.setStyleSheet(f"""
            QMainWindow, QWidget {{
                background-color: {PALETTE['base']};
                color: {PALETTE['text']};
                font-family: 'Segoe UI', 'Inter', 'SF Pro Display', sans-serif;
                font-size: 13px;
            }}
            QGroupBox {{
                background-color: {PALETTE['mantle']};
                border: 1px solid {PALETTE['surface0']};
                border-radius: 12px;
                margin-top: 26px;
                padding: 14px 14px 12px 14px;
                font-weight: 600;
                color: {PALETTE['subtext']};
                letter-spacing: 0.4px;
            }}
            QGroupBox::title {{
                subcontrol-origin: margin;
                subcontrol-position: top left;
                left: 14px;
                top: 2px;
                padding: 3px 10px;
                background-color: {PALETTE['surface0']};
                border-radius: 6px;
                color: {PALETTE['blue']};
                font-size: 12px;
            }}
            QPushButton {{
                background-color: {PALETTE['surface1']};
                color: {PALETTE['text']};
                border: none;
                border-radius: 8px;
                padding: 10px 18px;
                font-weight: 600;
            }}
            QPushButton:hover   {{ background-color: {PALETTE['surface2']}; }}
            QPushButton:pressed {{ background-color: {PALETTE['surface0']}; }}
            QPushButton:disabled {{
                background-color: {PALETTE['surface0']};
                color: {PALETTE['overlay']};
            }}
            QPushButton#primaryBtn {{
                background-color: {PALETTE['blue']};
                color: {PALETTE['crust']};
            }}
            QPushButton#primaryBtn:hover {{ background-color: {PALETTE['sky']}; }}
            QPushButton#dangerBtn {{
                background-color: transparent;
                color: {PALETTE['red']};
                border: 1px solid {PALETTE['red']};
            }}
            QPushButton#dangerBtn:hover {{
                background-color: rgba(243, 139, 168, 0.15);
            }}
            QLabel#imageLabel {{
                background-color: {PALETTE['crust']};
                border: 2px dashed {PALETTE['surface1']};
                border-radius: 12px;
                color: {PALETTE['overlay']};
                padding: 8px;
            }}
            QLabel#imageLabel[dragHover="true"] {{
                border-color: {PALETTE['blue']};
                color: {PALETTE['blue']};
            }}
            QLabel#statusLabel {{ color: {PALETTE['subtext']}; }}
            QLabel#badge {{
                border-radius: 6px;
                padding: 4px 10px;
                font-weight: 600;
                font-size: 12px;
            }}
            QLabel#badge[state="idle"]   {{ background-color: rgba(108,112,134,0.25); color: {PALETTE['overlay']}; }}
            QLabel#badge[state="loading"]{{ background-color: rgba(249,226,175,0.18);  color: {PALETTE['yellow']}; }}
            QLabel#badge[state="ok"]     {{ background-color: rgba(166,227,161,0.18);  color: {PALETTE['green']}; }}
            QLabel#badge[state="error"]  {{ background-color: rgba(243,139,168,0.18);  color: {PALETTE['red']}; }}
            QListWidget {{
                background-color: {PALETTE['crust']};
                border: 1px solid {PALETTE['surface0']};
                border-radius: 10px;
                padding: 6px;
                outline: 0;
            }}
            QListWidget::item {{
                padding: 6px 8px;
                border-radius: 6px;
            }}
            QListWidget::item:selected {{
                background-color: rgba(137, 180, 250, 0.20);
                color: {PALETTE['blue']};
            }}
            QProgressBar {{
                background-color: {PALETTE['crust']};
                border: none;
                border-radius: 6px;
                height: 10px;
                text-align: center;
                color: transparent;
            }}
            QProgressBar::chunk {{
                background-color: {PALETTE['blue']};
                border-radius: 6px;
            }}
            QSplitter::handle {{
                background-color: {PALETTE['surface0']};
                width: 2px;
            }}
        """)

    # ---------- UI ----------
    def _build_ui(self):
        central = QWidget()
        self.setCentralWidget(central)
        root = QVBoxLayout(central)
        root.setContentsMargins(18, 16, 18, 14)
        root.setSpacing(12)

        # ---------- Header ----------
        header = QHBoxLayout()
        header.setSpacing(14)

        brand_box = QVBoxLayout()
        brand_box.setSpacing(2)

        title = QLabel("DermaVision")
        title.setStyleSheet(f"""
            font-size: 22px;
            font-weight: 700;
            color: {PALETTE['blue']};
            letter-spacing: 0.5px;
        """)

        subtitle = QLabel("Skin Lesion Classification  ·  ConvNeXt-Tiny + Grad-CAM")
        subtitle.setStyleSheet(f"color: {PALETTE['overlay']}; font-size: 12px;")

        brand_box.addWidget(title)
        brand_box.addWidget(subtitle)
        header.addLayout(brand_box)
        header.addStretch(1)

        # Model status badge
        self.badge = QLabel("● No model loaded")
        self.badge.setObjectName("badge")
        self.badge.setProperty("state", "idle")
        self.badge.setAlignment(Qt.AlignCenter)
        self.badge.setMinimumWidth(160)
        header.addWidget(self.badge, 0, Qt.AlignVCenter)

        # Load weights button
        self.load_btn = QPushButton("  Load Model Weights…")
        self.load_btn.setObjectName("primaryBtn")
        self.load_btn.setCursor(Qt.PointingHandCursor)
        self.load_btn.clicked.connect(self.on_load_weights)
        header.addWidget(self.load_btn, 0, Qt.AlignVCenter)

        root.addLayout(header)

        # ---------- Splitter ----------
        splitter = QSplitter(Qt.Horizontal)
        splitter.setChildrenCollapsible(False)

        # ==================== LEFT: Visual Analysis ====================
        left_group = QGroupBox("Visual Analysis")
        left_layout = QVBoxLayout(left_group)
        left_layout.setSpacing(12)

        images_row = QHBoxLayout()
        images_row.setSpacing(10)

        # Original
        self.image_label = ImageDropLabel("Drop image here\nor click Open Image")
        self.image_label.setAlignment(Qt.AlignCenter)
        self.image_label.setObjectName("imageLabel")
        self.image_label.setMinimumSize(260, 260)
        self.image_label.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.image_label.fileDropped.connect(self.load_image)

        # Heatmap
        self.heatmap_label = QLabel("Heatmap Only")
        self.heatmap_label.setAlignment(Qt.AlignCenter)
        self.heatmap_label.setObjectName("imageLabel")
        self.heatmap_label.setMinimumSize(260, 260)
        self.heatmap_label.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)

        # Overlay
        self.overlay_label = QLabel("Grad-CAM Overlay")
        self.overlay_label.setAlignment(Qt.AlignCenter)
        self.overlay_label.setObjectName("imageLabel")
        self.overlay_label.setMinimumSize(260, 260)
        self.overlay_label.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)

        for w in (self.image_label, self.heatmap_label, self.overlay_label):
            images_row.addWidget(w)

        left_layout.addLayout(images_row, 1)

        # Action row
        action_row = QHBoxLayout()
        action_row.setSpacing(8)

        self.open_btn = QPushButton("  Open Image")
        self.open_btn.setCursor(Qt.PointingHandCursor)
        self.open_btn.clicked.connect(self.on_open_image)

        self.predict_btn = QPushButton("  Run Prediction")
        self.predict_btn.setObjectName("primaryBtn")
        self.predict_btn.setCursor(Qt.PointingHandCursor)
        self.predict_btn.clicked.connect(self.on_predict)
        self.predict_btn.setEnabled(False)

        self.clear_btn = QPushButton("  Clear")
        self.clear_btn.setObjectName("dangerBtn")
        self.clear_btn.setCursor(Qt.PointingHandCursor)
        self.clear_btn.clicked.connect(self.on_clear)

        action_row.addWidget(self.open_btn)
        action_row.addStretch(1)
        action_row.addWidget(self.clear_btn)
        action_row.addWidget(self.predict_btn)
        left_layout.addLayout(action_row)

        splitter.addWidget(left_group)

        # ==================== RIGHT: Results ====================
        right_group = QGroupBox("Prediction Results")
        right_layout = QVBoxLayout(right_group)
        right_layout.setSpacing(12)

        # Top prediction card
        self.top_card = QFrame()
        self.top_card.setStyleSheet(f"""
            QFrame {{
                background-color: {PALETTE['crust']};
                border: 1px solid {PALETTE['surface0']};
                border-radius: 12px;
            }}
        """)
        top_layout = QVBoxLayout(self.top_card)
        top_layout.setContentsMargins(16, 14, 16, 14)
        top_layout.setSpacing(6)

        top_caption = QLabel("TOP PREDICTION")
        top_caption.setStyleSheet(
            f"color: {PALETTE['overlay']}; font-size: 11px; letter-spacing: 1.2px; font-weight: 600;"
        )
        top_layout.addWidget(top_caption)

        self.top_label = QLabel("—")
        self.top_label.setStyleSheet(
            f"font-size: 22px; font-weight: 700; color: {PALETTE['green']};"
        )
        top_layout.addWidget(self.top_label)

        self.conf_bar = QProgressBar()
        self.conf_bar.setRange(0, 100)
        self.conf_bar.setValue(0)
        self.conf_bar.setTextVisible(False)
        self.conf_bar.setFixedHeight(8)
        top_layout.addWidget(self.conf_bar)

        self.conf_text = QLabel("")
        self.conf_text.setStyleSheet(f"color: {PALETTE['subtext']}; font-size: 12px;")
        top_layout.addWidget(self.conf_text)

        right_layout.addWidget(self.top_card)

        # Chart
        chart_label = QLabel("Probability Distribution")
        chart_label.setStyleSheet(
            f"color: {PALETTE['subtext']}; font-weight: 600; margin-top: 4px;"
        )
        right_layout.addWidget(chart_label)

        self.figure = Figure(figsize=(4, 2.8), dpi=90, facecolor=PALETTE['mantle'])
        self.canvas = FigureCanvas(self.figure)
        self.canvas.setStyleSheet(f"background-color: {PALETTE['mantle']};")
        self.canvas.setMinimumHeight(220)
        right_layout.addWidget(self.canvas, 1)

        # Class list
        list_label = QLabel("All Class Probabilities")
        list_label.setStyleSheet(
            f"color: {PALETTE['subtext']}; font-weight: 600;"
        )
        right_layout.addWidget(list_label)

        self.prob_list = QListWidget()
        self.prob_list.setMinimumHeight(140)
        right_layout.addWidget(self.prob_list, 1)

        splitter.addWidget(right_group)
        splitter.setSizes([900, 500])
        splitter.setStretchFactor(0, 3)
        splitter.setStretchFactor(1, 2)

        root.addWidget(splitter, 1)

        # ---------- Status bar ----------
        status_row = QHBoxLayout()
        status_row.setContentsMargins(2, 0, 2, 0)

        self.status_bar = QLabel("Ready")
        self.status_bar.setObjectName("statusLabel")
        status_row.addWidget(self.status_bar, 1)

        self.ready_indicator = QLabel("  Ready")
        self.ready_indicator.setStyleSheet(
            f"color: {PALETTE['green']}; font-weight: 600; font-size: 12px;"
        )
        status_row.addWidget(self.ready_indicator, 0, Qt.AlignRight)

        root.addLayout(status_row)

    # ---------- Badge helper ----------
    def _set_badge(self, text, state):
        self.badge.setText(text)
        self.badge.setProperty("state", state)
        self.badge.style().unpolish(self.badge)
        self.badge.style().polish(self.badge)

    # ---------- Model loading ----------
    def on_load_weights(self):
        path, _ = QFileDialog.getOpenFileName(
            self, "Select Model Weights", "",
            "Keras Weights (*.h5 *.keras *.weights.h5);;All Files (*)"
        )
        if not path:
            return

        self.load_btn.setEnabled(False)
        self._set_badge("● Loading model…", "loading")
        self.status_bar.setText(f"Loading weights: {os.path.basename(path)} …")

        self._loader = ModelLoaderThread(path)
        self._loader.finished.connect(self._on_model_loaded)
        self._loader.error.connect(self._on_model_error)
        self._loader.start()

    def _on_model_loaded(self, model):
        self.model = model
        self.load_btn.setEnabled(True)
        self._set_badge("● Model ready", "ok")
        self.status_bar.setText("Model loaded — open an image to begin.")
        self._update_predict_btn()
        QMessageBox.information(self, "Success", "Model weights loaded successfully.")

    def _on_model_error(self, err):
        self.load_btn.setEnabled(True)
        self._set_badge("● Model failed", "error")
        self.status_bar.setText("Model loading failed.")
        QMessageBox.critical(self, "Model Loading Error", err)

    # ---------- Image ----------
    def on_open_image(self):
        path, _ = QFileDialog.getOpenFileName(
            self, "Select Image", "",
            "Images (*.png *.jpg *.jpeg *.bmp)"
        )
        if path:
            self.load_image(path)

    def load_image(self, path):
        try:
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                raise ValueError("Unable to read the image.")
            self.current_img_path = path
            self.current_img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            self._show_images(self.current_img_rgb, None)
            self.status_bar.setText(f"Loaded: {os.path.basename(path)}")
            self._update_predict_btn()
            self._clear_results()
        except Exception as e:
            QMessageBox.warning(self, "Error", str(e))

    def _show_images(self, img_rgb, heatmap):
        h, w, ch = img_rgb.shape

        # Original
        qimg = QImage(img_rgb.data, w, h, ch * w, QImage.Format_RGB888)
        pix  = QPixmap.fromImage(qimg)
        self.image_label.setPixmap(
            pix.scaled(self.image_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation)
        )

        if heatmap is None:
            self.heatmap_label.setText("Heatmap Only")
            self.overlay_label.setText("Grad-CAM Overlay")
            return

        heatmap_resized = cv2.resize(heatmap, (w, h))

        # Heatmap only
        heatmap_color = cv2.applyColorMap(
            np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
        )
        heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        qimg_heat = QImage(heatmap_rgb.data, w, h, ch * w, QImage.Format_RGB888)
        pix_heat  = QPixmap.fromImage(qimg_heat)
        self.heatmap_label.setPixmap(
            pix_heat.scaled(self.heatmap_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation)
        )

        # Overlay
        overlay = cv2.addWeighted(img_rgb, 0.6, heatmap_rgb, 0.4, 0)
        qimg_cam = QImage(overlay.data, w, h, ch * w, QImage.Format_RGB888)
        pix_cam  = QPixmap.fromImage(qimg_cam)
        self.overlay_label.setPixmap(
            pix_cam.scaled(self.overlay_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation)
        )

    def _update_predict_btn(self):
        self.predict_btn.setEnabled(
            self.model is not None and self.current_img_rgb is not None
        )

    # ---------- Predict ----------
    def on_predict(self):
        if self.model is None or self.current_img_rgb is None:
            return
        self.predict_btn.setEnabled(False)
        self.open_btn.setEnabled(False)
        self.status_bar.setText("Running prediction and computing Grad-CAM …")
        self.ready_indicator.setText("  Processing")
        self.ready_indicator.setStyleSheet(
            f"color: {PALETTE['yellow']}; font-weight: 600; font-size: 12px;"
        )

        self._predictor = PredictThread(
            self.model, self.current_img_rgb, self.current_img_path
        )
        self._predictor.finished.connect(self._on_predicted)
        self._predictor.error.connect(self._on_predict_error)
        self._predictor.start()

    def _on_predicted(self, probs, heatmap, img_bgr):
        self._update_results(probs)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        self._show_images(img_rgb, heatmap)

        self.predict_btn.setEnabled(True)
        self.open_btn.setEnabled(True)

        top_idx = int(np.argmax(probs))
        self.status_bar.setText(
            f"Prediction complete — {CLASS_NAMES[top_idx]}  "
            f"({probs[top_idx]*100:.2f}% confidence)"
        )
        self.ready_indicator.setText("  Ready")
        self.ready_indicator.setStyleSheet(
            f"color: {PALETTE['green']}; font-weight: 600; font-size: 12px;"
        )

    def _on_predict_error(self, err):
        self.predict_btn.setEnabled(True)
        self.open_btn.setEnabled(True)
        self.status_bar.setText("Prediction failed.")
        self.ready_indicator.setText("  Error")
        self.ready_indicator.setStyleSheet(
            f"color: {PALETTE['red']}; font-weight: 600; font-size: 12px;"
        )
        QMessageBox.critical(self, "Prediction Error", err)

    # ---------- Results ----------
    def _clear_results(self):
        self.top_label.setText("—")
        self.conf_text.setText("")
        self.conf_bar.setValue(0)
        self.prob_list.clear()
        self.figure.clear()
        self.canvas.draw()
        self.heatmap_label.setText("Heatmap Only")
        self.overlay_label.setText("Grad-CAM Overlay")

    def _update_results(self, probs):
        top_idx = int(np.argmax(probs))
        pct     = probs[top_idx] * 100

        self.top_label.setText(CLASS_NAMES[top_idx])
        self.conf_bar.setValue(int(round(pct)))
        self.conf_text.setText(f"Confidence: {pct:.2f}%")

        # Class list
        self.prob_list.clear()
        for i in np.argsort(probs)[::-1]:
            name = CLASS_NAMES[i]
            p    = probs[i] * 100
            item = QListWidgetItem(f"{name:<16s}   {p:6.2f}%")
            item.setForeground(
                QColor(PALETTE['blue']) if i == top_idx else QColor(PALETTE['text'])
            )
            if i == top_idx:
                f = item.font()
                f.setBold(True)
                item.setFont(f)
            self.prob_list.addItem(item)

        # Bar chart
        self.figure.clear()
        ax = self.figure.add_subplot(111)
        ax.set_facecolor(PALETTE['mantle'])
        colors = [
            PALETTE['blue'] if i == top_idx else PALETTE['surface2']
            for i in range(len(probs))
        ]
        ax.bar(CLASS_NAMES, probs * 100, color=colors, edgecolor=PALETTE['crust'])
        ax.set_ylim(0, 100)
        ax.set_ylabel("Probability (%)", color=PALETTE['text'], fontsize=9)
        ax.tick_params(colors=PALETTE['subtext'], labelsize=8)
        for s in ax.spines.values():
            s.set_color(PALETTE['surface1'])
        ax.grid(axis="y", alpha=0.15, color=PALETTE['text'])
        self.figure.tight_layout()
        self.canvas.draw()

    # ---------- Clear ----------
    def on_clear(self):
        self.current_img_rgb  = None
        self.current_img_path = None
        self.image_label.setPixmap(QPixmap())
        self.image_label.setText("Drop image here\nor click Open Image")
        self._clear_results()
        self._update_predict_btn()
        self.status_bar.setText("Cleared.")
        self.ready_indicator.setText("  Ready")
        self.ready_indicator.setStyleSheet(
            f"color: {PALETTE['green']}; font-weight: 600; font-size: 12px;"
        )


def main():
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    win = SkinClassifierApp()
    win.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()

SystemExit: 0

In [ ]:
# skin_app_simple.py
import sys
import os
import cv2
import time
import numpy as np
import tensorflow as tf
from PyQt5.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QFileDialog,
    QVBoxLayout, QTextEdit, QHBoxLayout, QMessageBox
)
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import Qt

from model_builder import (
    load_model_from_weights, predict_image, extract_handcrafted_features_bgr,
    CLASS_NAMES, NUM_CLASSES, HANDCRAFT_DIM, IMG_SIZE
)
from tensorflow.keras.applications.efficientnet import preprocess_input as ef_preprocess


# ======================= Full Class Names =======================
FULL_CLASS_NAMES = {
    "akiec": "Actinic Keratosis / Bowen's Disease",
    "bcc":   "Basal Cell Carcinoma",
    "bkl":   "Benign Keratosis",
    "df":    "Dermatofibroma",
    "mel":   "Melanoma",
    "nv":    "Melanocytic Nevus",
    "vasc":  "Vascular Lesion",
}


def get_full_name(short_name):
    """Convert short class code (e.g. 'bkl') to full readable name."""
    return FULL_CLASS_NAMES.get(short_name.lower(), short_name)


# ======================= Grad-CAM =======================
def get_gradcam_heatmap(model, img_array, hand_features=None,
                        pred_index=None,
                        layer_name='convnext_tiny_stage_3_block_2_pointwise_conv_2'):
    grad_model = None
    try:
        grad_model = tf.keras.models.Model(
            model.inputs,
            [model.get_layer(layer_name).output, model.output]
        )
    except Exception:
        for layer in reversed(model.layers):
            if 'conv' in layer.name and 'pointwise' in layer.name:
                grad_model = tf.keras.models.Model(
                    model.inputs, [layer.output, model.output]
                )
                break

    if grad_model is None:
        raise ValueError(f"Could not find layer '{layer_name}' or an alternative.")

    inputs = [img_array]
    if hand_features is not None:
        inputs.append(hand_features)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


# ======================= Main App =======================
class SkinLesionApp(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("DermaVision : Skin Lesion Classifier")
        self.setGeometry(100, 100, 1300, 800)

        # State
        self.model              = None
        self.current_image_path = ""
        self.current_img_rgb    = None
        self.current_img_bgr    = None

        # Layouts
        self.layout = QVBoxLayout()
        self.image_layout = QHBoxLayout()

        # Title
        self.label = QLabel("DermaVision : Skin Lesion Classification with Grad-CAM")
        self.label.setAlignment(Qt.AlignCenter)
        self.label.setStyleSheet("font-size: 20px; font-weight: bold;")
        self.layout.addWidget(self.label)

        # Buttons row
        btn_row = QHBoxLayout()
        self.load_model_button = QPushButton("Load Model Weights")
        self.load_model_button.clicked.connect(self.browse_model)
        self.browse_button = QPushButton("Browse Skin Image")
        self.browse_button.clicked.connect(self.browse_image)
        btn_row.addWidget(self.load_model_button)
        btn_row.addWidget(self.browse_button)
        self.layout.addLayout(btn_row)

        # Result Display
        self.result_text = QTextEdit()
        self.result_text.setReadOnly(True)
        self.result_text.setStyleSheet("font-size: 16px;")
        self.result_text.setFixedHeight(200)
        self.layout.addWidget(self.result_text)

        # Image Labels
        self.original_label = QLabel("Original Image")
        self.heatmap_label  = QLabel("Heatmap")
        self.overlay_label  = QLabel("Grad-CAM Overlay")
        for lbl in [self.original_label, self.heatmap_label, self.overlay_label]:
            lbl.setFixedSize(400, 400)
            lbl.setAlignment(Qt.AlignCenter)
            lbl.setStyleSheet("border: 1px solid black;")
        self.image_layout.addWidget(self.original_label)
        self.image_layout.addWidget(self.heatmap_label)
        self.image_layout.addWidget(self.overlay_label)
        self.layout.addLayout(self.image_layout)

        self.setLayout(self.layout)

    # ---------- Model ----------
    def browse_model(self):
        path, _ = QFileDialog.getOpenFileName(
            self, "Select Model Weights", "",
            "Keras Weights (*.h5 *.keras *.weights.h5);;All Files (*)"
        )
        if not path:
            return
        try:
            self.result_text.setText(f"Loading model: {os.path.basename(path)} …")
            QApplication.processEvents()
            self.model = load_model_from_weights(path)
            self.result_text.setText(f"✅ Model loaded: {os.path.basename(path)}")
        except Exception as e:
            QMessageBox.critical(self, "Model Loading Error", str(e))
            self.result_text.setText("❌ Model loading failed.")

    # ---------- Image ----------
    def browse_image(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Skin Image", "",
            "Images (*.png *.jpg *.jpeg *.bmp)"
        )
        if file_path:
            self.current_image_path = file_path
            self.load_image(file_path)

    def load_image(self, file_path):
        img_bgr = cv2.imread(file_path)
        if img_bgr is None:
            QMessageBox.warning(self, "Error", "Unable to read the image.")
            return
        self.current_img_bgr = img_bgr
        self.current_img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        self.display_image(self.current_img_rgb, self.original_label)
        self.heatmap_label.setText("Heatmap")
        self.overlay_label.setText("Grad-CAM Overlay")
        self.result_text.setText(f"Loaded: {os.path.basename(file_path)}\n"
                                 "Press 'Run Prediction' to analyze.")

        # auto-run if model loaded
        if self.model is not None:
            self.run_model()

    # ---------- Prediction ----------
    def run_model(self):
        if self.model is None:
            QMessageBox.warning(self, "No Model", "Please load model weights first.")
            return
        if self.current_img_rgb is None:
            QMessageBox.warning(self, "No Image", "Please load an image first.")
            return

        start_time = time.time()

        # ---- Predict ----
        probs = predict_image(self.model, self.current_img_rgb)

        # ---- Grad-CAM ----
        img_resized = cv2.resize(self.current_img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
        img_pre     = ef_preprocess(img_resized)[np.newaxis, ...]
        hand_feat   = extract_handcrafted_features_bgr(self.current_img_bgr)[np.newaxis, ...]
        heatmap     = get_gradcam_heatmap(
            self.model, img_pre, hand_feat,
            pred_index=int(np.argmax(probs))
        )

        # ---- Text result (with FULL names) ----
        top_idx = int(np.argmax(probs))
        top_full = get_full_name(CLASS_NAMES[top_idx])

        lines = [
            f"Prediction : {top_full} ({CLASS_NAMES[top_idx]})",
            f"Confidence : {probs[top_idx]*100:.2f}%\n",
            "All Probabilities:"
        ]
        for i in np.argsort(probs)[::-1]:
            full_name = get_full_name(CLASS_NAMES[i])
            lines.append(
                f"  {full_name:<42s} {probs[i]*100:6.2f}%   ({CLASS_NAMES[i]})"
            )

        duration = time.time() - start_time
        lines.append(f"\n🕒 Analysis Time: {duration:.2f} seconds")
        self.result_text.setText("\n".join(lines))

        # ---- Images ----
        self.display_image(self.current_img_rgb, self.original_label)

        h, w, ch = self.current_img_rgb.shape
        heatmap_resized = cv2.resize(heatmap, (w, h))
        heatmap_color = cv2.applyColorMap(
            np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
        )
        heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(self.current_img_rgb, 0.6, heatmap_rgb, 0.4, 0)

        self.display_image(heatmap_rgb, self.heatmap_label)
        self.display_image(overlay, self.overlay_label)

    # ---------- Display helper ----------
    def display_image(self, image_rgb, label):
        image_resized = cv2.resize(image_rgb, (400, 400))
        height, width, channel = image_resized.shape
        bytes_per_line = 3 * width
        q_img = QImage(image_resized.data, width, height, bytes_per_line,
                       QImage.Format_RGB888)
        pixmap = QPixmap.fromImage(q_img)
        label.setPixmap(pixmap)


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = SkinLesionApp()
    window.show()
    sys.exit(app.exec_())

In [1]:
1

1

In [ ]:
# skin_app_simple.py
import sys
import os
import cv2
import time
import numpy as np
import tensorflow as tf
from PyQt5.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QFileDialog,
    QVBoxLayout, QTextEdit, QHBoxLayout, QMessageBox
)
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import Qt

from model_builder import (
    load_model_from_weights, predict_image, extract_handcrafted_features_bgr,
    CLASS_NAMES, NUM_CLASSES, HANDCRAFT_DIM, IMG_SIZE
)
from tensorflow.keras.applications.efficientnet import preprocess_input as ef_preprocess


# ======================= Full Class Names =======================
FULL_CLASS_NAMES = {
    "akiec": "Actinic Keratosis / Bowen's Disease",
    "bcc":   "Basal Cell Carcinoma",
    "bkl":   "Benign Keratosis",
    "df":    "Dermatofibroma",
    "mel":   "Melanoma",
    "nv":    "Melanocytic Nevus",
    "vasc":  "Vascular Lesion",
}


def get_full_name(short_name):
    """Convert short class code (e.g. 'bkl') to full readable name."""
    return FULL_CLASS_NAMES.get(short_name.lower(), short_name)


# ======================= Grad-CAM =======================
def get_gradcam_heatmap(model, img_array, hand_features=None,
                        pred_index=None,
                        layer_name='convnext_tiny_stage_3_block_2_pointwise_conv_2'):
    grad_model = None
    try:
        grad_model = tf.keras.models.Model(
            model.inputs,
            [model.get_layer(layer_name).output, model.output]
        )
    except Exception:
        for layer in reversed(model.layers):
            if 'conv' in layer.name and 'pointwise' in layer.name:
                grad_model = tf.keras.models.Model(
                    model.inputs, [layer.output, model.output]
                )
                break

    if grad_model is None:
        raise ValueError(f"Could not find layer '{layer_name}' or an alternative.")

    inputs = [img_array]
    if hand_features is not None:
        inputs.append(hand_features)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


# ======================= Main App =======================
class SkinLesionApp(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("DermaVision : Skin Lesion Classifier")
        self.setGeometry(100, 100, 1300, 800)

        # State
        self.model              = None
        self.current_image_path = ""
        self.current_img_rgb    = None
        self.current_img_bgr    = None

        # Layouts
        self.layout = QVBoxLayout()
        self.image_layout = QHBoxLayout()

        # Title
        self.label = QLabel("DermaVision : Skin Lesion Classification with Grad-CAM")
        self.label.setAlignment(Qt.AlignCenter)
        self.label.setStyleSheet("font-size: 20px; font-weight: bold;")
        self.layout.addWidget(self.label)

        # Buttons row
        btn_row = QHBoxLayout()
        self.load_model_button = QPushButton("Load Model Weights")
        self.load_model_button.clicked.connect(self.browse_model)
        self.browse_button = QPushButton("Browse Skin Image")
        self.browse_button.clicked.connect(self.browse_image)
        btn_row.addWidget(self.load_model_button)
        btn_row.addWidget(self.browse_button)
        self.layout.addLayout(btn_row)

        # Result Display
        self.result_text = QTextEdit()
        self.result_text.setReadOnly(True)
        self.result_text.setStyleSheet("font-size: 16px;")
        self.result_text.setFixedHeight(200)
        self.layout.addWidget(self.result_text)

        # Image Labels
        self.original_label = QLabel("Original Image")
        self.heatmap_label  = QLabel("Heatmap")
        self.overlay_label  = QLabel("Grad-CAM Overlay")
        for lbl in [self.original_label, self.heatmap_label, self.overlay_label]:
            lbl.setFixedSize(400, 400)
            lbl.setAlignment(Qt.AlignCenter)
            lbl.setStyleSheet("border: 1px solid black;")
        self.image_layout.addWidget(self.original_label)
        self.image_layout.addWidget(self.heatmap_label)
        self.image_layout.addWidget(self.overlay_label)
        self.layout.addLayout(self.image_layout)

        self.setLayout(self.layout)

    # ---------- Model ----------
    def browse_model(self):
        path, _ = QFileDialog.getOpenFileName(
            self, "Select Model Weights", "",
            "Keras Weights (*.h5 *.keras *.weights.h5);;All Files (*)"
        )
        if not path:
            return
        try:
            self.result_text.setText(f"Loading model: {os.path.basename(path)} …")
            QApplication.processEvents()
            self.model = load_model_from_weights(path)
            self.result_text.setText(f"✅ Model loaded: {os.path.basename(path)}")
        except Exception as e:
            QMessageBox.critical(self, "Model Loading Error", str(e))
            self.result_text.setText("❌ Model loading failed.")

    # ---------- Image ----------
    def browse_image(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Skin Image", "",
            "Images (*.png *.jpg *.jpeg *.bmp)"
        )
        if file_path:
            self.current_image_path = file_path
            self.load_image(file_path)

    def load_image(self, file_path):
        img_bgr = cv2.imread(file_path)
        if img_bgr is None:
            QMessageBox.warning(self, "Error", "Unable to read the image.")
            return
        self.current_img_bgr = img_bgr
        self.current_img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        self.display_image(self.current_img_rgb, self.original_label)
        self.heatmap_label.setText("Heatmap")
        self.overlay_label.setText("Grad-CAM Overlay")
        self.result_text.setText(f"Loaded: {os.path.basename(file_path)}\n"
                                 "Press 'Run Prediction' to analyze.")

        # auto-run if model loaded
        if self.model is not None:
            self.run_model()

    # ---------- Prediction ----------
    def run_model(self):
        if self.model is None:
            QMessageBox.warning(self, "No Model", "Please load model weights first.")
            return
        if self.current_img_rgb is None:
            QMessageBox.warning(self, "No Image", "Please load an image first.")
            return

        start_time = time.time()

        # ---- Predict ----
        probs = predict_image(self.model, self.current_img_rgb)

        # ---- Grad-CAM ----
        img_resized = cv2.resize(self.current_img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
        img_pre     = ef_preprocess(img_resized)[np.newaxis, ...]
        hand_feat   = extract_handcrafted_features_bgr(self.current_img_bgr)[np.newaxis, ...]
        heatmap     = get_gradcam_heatmap(
            self.model, img_pre, hand_feat,
            pred_index=int(np.argmax(probs))
        )

        # ---- Text result (with FULL names) ----
        top_idx = int(np.argmax(probs))
        top_full = get_full_name(CLASS_NAMES[top_idx])

        lines = [
            f"Prediction : {top_full} ({CLASS_NAMES[top_idx]})",
            f"Confidence : {probs[top_idx]*100:.2f}%\n",
            "All Probabilities:"
        ]
        for i in np.argsort(probs)[::-1]:
            full_name = get_full_name(CLASS_NAMES[i])
            lines.append(
                f"  {full_name:<42s} {probs[i]*100:6.2f}%   ({CLASS_NAMES[i]})"
            )

        duration = time.time() - start_time
        lines.append(f"\n🕒 Analysis Time: {duration:.2f} seconds")
        self.result_text.setText("\n".join(lines))

        # ---- Images ----
        self.display_image(self.current_img_rgb, self.original_label)

        h, w, ch = self.current_img_rgb.shape
        heatmap_resized = cv2.resize(heatmap, (w, h))
        heatmap_color = cv2.applyColorMap(
            np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
        )
        heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(self.current_img_rgb, 0.6, heatmap_rgb, 0.4, 0)

        self.display_image(heatmap_rgb, self.heatmap_label)
        self.display_image(overlay, self.overlay_label)

    # ---------- Display helper ----------
    def display_image(self, image_rgb, label):
        image_resized = cv2.resize(image_rgb, (400, 400))
        height, width, channel = image_resized.shape
        bytes_per_line = 3 * width
        q_img = QImage(image_resized.data, width, height, bytes_per_line,
                       QImage.Format_RGB888)
        pixmap = QPixmap.fromImage(q_img)
        label.setPixmap(pixmap)


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = SkinLesionApp()
    window.show()
    sys.exit(app.exec_())

C:\Users\Lenovo\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\Lenovo\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\Lenovo\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


C:\Users\Lenovo\anaconda3\Lib\site-packages\skimage\feature\texture.py:360: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


In [1]:
# skin_app_simple.py
import sys
import os
import cv2
import time
import numpy as np
import tensorflow as tf
from PyQt5.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QFileDialog,
    QVBoxLayout, QTextEdit, QHBoxLayout, QMessageBox
)
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import Qt

from model_builder import (
    load_model_from_weights, predict_image, extract_handcrafted_features_bgr,
    CLASS_NAMES, NUM_CLASSES, HANDCRAFT_DIM, IMG_SIZE
)
from tensorflow.keras.applications.efficientnet import preprocess_input as ef_preprocess


# ======================= Full Class Names =======================
FULL_CLASS_NAMES = {
    "akiec": "Actinic Keratosis",
    "bcc":   "Basal Cell Carcinoma",
    "bkl":   "Benign Keratosis",
    "df":    "Dermatofibroma",
    "mel":   "Melanoma",
    "nv":    "Melanocytic Nevus",
    "vasc":  "Vascular Lesion",
}


def get_full_name(short_name):
    """Convert short class code (e.g. 'bkl') to full readable name."""
    return FULL_CLASS_NAMES.get(short_name.lower(), short_name)


# ======================= Grad-CAM =======================
def get_gradcam_heatmap(model, img_array, hand_features=None,
                        pred_index=None,
                        layer_name='convnext_tiny_stage_3_block_2_pointwise_conv_2'):
    grad_model = None
    try:
        grad_model = tf.keras.models.Model(
            model.inputs,
            [model.get_layer(layer_name).output, model.output]
        )
    except Exception:
        for layer in reversed(model.layers):
            if 'conv' in layer.name and 'pointwise' in layer.name:
                grad_model = tf.keras.models.Model(
                    model.inputs, [layer.output, model.output]
                )
                break

    if grad_model is None:
        raise ValueError(f"Could not find layer '{layer_name}' or an alternative.")

    inputs = [img_array]
    if hand_features is not None:
        inputs.append(hand_features)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


# ======================= Main App =======================
class SkinLesionApp(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("ConvNeXt-Tiny Deep Handcrafted")
        self.setGeometry(100, 100, 1300, 850)

        # State
        self.model              = None
        self.current_image_path = ""
        self.current_img_rgb    = None
        self.current_img_bgr    = None

        # Layouts
        self.layout = QVBoxLayout()
        self.image_layout = QHBoxLayout()

        # Title
        self.label = QLabel("ConvNeXt-Tiny Deep Handcrafted Skin Lesion Classification with Grad-CAM")
        self.label.setAlignment(Qt.AlignCenter)
        self.label.setStyleSheet("font-size: 20px; font-weight: bold;")
        self.layout.addWidget(self.label)

        # Buttons row
        btn_row = QHBoxLayout()
        self.load_model_button = QPushButton("Load Model Weights")
        self.load_model_button.clicked.connect(self.browse_model)
        self.browse_button = QPushButton("Browse Skin Image")
        self.browse_button.clicked.connect(self.browse_image)
        btn_row.addWidget(self.load_model_button)
        btn_row.addWidget(self.browse_button)
        self.layout.addLayout(btn_row)

        # Result Display (QTextEdit بدون Scroll handles + ارتفاع ديناميكي)
        self.result_text = QTextEdit()
        self.result_text.setReadOnly(True)
        self.result_text.setStyleSheet("font-size: 16px;")
        self.result_text.setLineWrapMode(QTextEdit.WidgetWidth)
        self.result_text.setVerticalScrollBarPolicy(Qt.ScrollBarAlwaysOff)
        self.result_text.setHorizontalScrollBarPolicy(Qt.ScrollBarAlwaysOff)
        self.result_text.setFixedHeight(320)   # ارتفاع ابتدائي
        self.layout.addWidget(self.result_text)

        # Image Labels
        self.original_label = QLabel("Original Image")
        self.heatmap_label  = QLabel("Heatmap")
        self.overlay_label  = QLabel("Grad-CAM Overlay")
        for lbl in [self.original_label, self.heatmap_label, self.overlay_label]:
            lbl.setFixedSize(400, 400)
            lbl.setAlignment(Qt.AlignCenter)
            lbl.setStyleSheet("border: 1px solid black;")
        self.image_layout.addWidget(self.original_label)
        self.image_layout.addWidget(self.heatmap_label)
        self.image_layout.addWidget(self.overlay_label)
        self.layout.addLayout(self.image_layout)

        self.setLayout(self.layout)

    # ---------- Helper: set result text with auto height ----------
    def set_result_text(self, text):
        """Sets the result text and auto-adjusts the widget height to fit all lines."""
        self.result_text.setText(text)
        # احسب ارتفاع المحتوى بعد ضبط النص
        doc_height = self.result_text.document().size().height()
        # حد أدنى 120، وحد أقصى حسب مساحة الشاشة
        new_height = int(doc_height) + 20
        self.result_text.setFixedHeight(max(120, new_height))

    # ---------- Model ----------
    def browse_model(self):
        path, _ = QFileDialog.getOpenFileName(
            self, "Select Model Weights", "",
            "Keras Weights (*.h5 *.keras *.weights.h5);;All Files (*)"
        )
        if not path:
            return
        try:
            self.set_result_text(f"Loading model: {os.path.basename(path)} …")
            QApplication.processEvents()
            self.model = load_model_from_weights(path)
            self.set_result_text(f"✅ Model loaded: {os.path.basename(path)}")
        except Exception as e:
            QMessageBox.critical(self, "Model Loading Error", str(e))
            self.set_result_text("❌ Model loading failed.")

    # ---------- Image ----------
    def browse_image(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Skin Image", "",
            "Images (*.png *.jpg *.jpeg *.bmp)"
        )
        if file_path:
            self.current_image_path = file_path
            self.load_image(file_path)

    def load_image(self, file_path):
        img_bgr = cv2.imread(file_path)
        if img_bgr is None:
            QMessageBox.warning(self, "Error", "Unable to read the image.")
            return
        self.current_img_bgr = img_bgr
        self.current_img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        self.display_image(self.current_img_rgb, self.original_label)
        self.heatmap_label.setText("Heatmap")
        self.overlay_label.setText("Grad-CAM Overlay")
        self.set_result_text(
            f"Loaded: {os.path.basename(file_path)}\n"
            "Press 'Run Prediction' to analyze."
        )

        # auto-run if model loaded
        if self.model is not None:
            self.run_model()

    # ---------- Prediction ----------
    def run_model(self):
        if self.model is None:
            QMessageBox.warning(self, "No Model", "Please load model weights first.")
            return
        if self.current_img_rgb is None:
            QMessageBox.warning(self, "No Image", "Please load an image first.")
            return

        start_time = time.time()

        # ---- Predict ----
        probs = predict_image(self.model, self.current_img_rgb)

        # ---- Grad-CAM ----
        img_resized = cv2.resize(self.current_img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
        img_pre     = ef_preprocess(img_resized)[np.newaxis, ...]
        hand_feat   = extract_handcrafted_features_bgr(self.current_img_bgr)[np.newaxis, ...]
        heatmap     = get_gradcam_heatmap(
            self.model, img_pre, hand_feat,
            pred_index=int(np.argmax(probs))
        )

        # ---- Text result (with FULL names) ----
        top_idx = int(np.argmax(probs))
        top_full = get_full_name(CLASS_NAMES[top_idx])

        lines = [
            f"Prediction : {top_full} ({CLASS_NAMES[top_idx]})",
            f"Confidence : {probs[top_idx]*100:.2f}%\n",
            "All Probabilities:"
        ]
        for i in np.argsort(probs)[::-1]:
            full_name = get_full_name(CLASS_NAMES[i])
            lines.append(
                f"  {full_name:<42s} {probs[i]*100:6.2f}%   ({CLASS_NAMES[i]})"
            )

        duration = time.time() - start_time
        lines.append(f"\n🕒 Analysis Time: {duration:.2f} seconds")
        self.set_result_text("\n".join(lines))

        # ---- Images ----
        self.display_image(self.current_img_rgb, self.original_label)

        h, w, ch = self.current_img_rgb.shape
        heatmap_resized = cv2.resize(heatmap, (w, h))
        heatmap_color = cv2.applyColorMap(
            np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
        )
        heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(self.current_img_rgb, 0.6, heatmap_rgb, 0.4, 0)

        self.display_image(heatmap_rgb, self.heatmap_label)
        self.display_image(overlay, self.overlay_label)

    # ---------- Display helper ----------
    def display_image(self, image_rgb, label):
        image_resized = cv2.resize(image_rgb, (400, 400))
        height, width, channel = image_resized.shape
        bytes_per_line = 3 * width
        q_img = QImage(image_resized.data, width, height, bytes_per_line,
                       QImage.Format_RGB888)
        pixmap = QPixmap.fromImage(q_img)
        label.setPixmap(pixmap)


if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = SkinLesionApp()
    window.show()
    sys.exit(app.exec_())

C:\Users\Lenovo\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\Lenovo\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\Lenovo\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


SystemExit: 0

C:\Users\Lenovo\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# skin_app_simple.py
import sys
import os
import cv2
import time
import numpy as np
import tensorflow as tf
from PyQt5.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QFileDialog,
    QVBoxLayout, QHBoxLayout, QMessageBox, QFrame, QProgressBar,
    QGraphicsDropShadowEffect, QSizePolicy, QScrollArea
)
from PyQt5.QtGui import QPixmap, QImage, QFont, QColor
from PyQt5.QtCore import Qt, QSize

from model_builder import (
    load_model_from_weights, predict_image, extract_handcrafted_features_bgr,
    CLASS_NAMES, NUM_CLASSES, HANDCRAFT_DIM, IMG_SIZE
)
from tensorflow.keras.applications.efficientnet import preprocess_input as ef_preprocess


# ======================= Full Class Names =======================
FULL_CLASS_NAMES = {
    "akiec": "Actinic Keratosis",
    "bcc":   "Basal Cell Carcinoma",
    "bkl":   "Benign Keratosis",
    "df":    "Dermatofibroma",
    "mel":   "Melanoma",
    "nv":    "Melanocytic Nevus",
    "vasc":  "Vascular Lesion",
}


def get_full_name(short_name):
    return FULL_CLASS_NAMES.get(short_name.lower(), short_name)


# ======================= Light Theme (Journal-Ready) =======================
class Theme:
    BG          = "#ffffff"
    BG_CARD     = "#ffffff"
    BG_SUBTLE   = "#f7f8fa"
    BORDER      = "#e1e4e8"
    BORDER_DARK = "#cbd1d9"
    TEXT        = "#1a1f2e"
    TEXT_DIM    = "#6b7280"
    ACCENT      = "#2563eb"
    ACCENT_LIGHT= "#3b82f6"
    ACCENT_SOFT = "#eff6ff"
    SUCCESS     = "#059669"
    SUCCESS_SOFT= "#ecfdf5"
    WARNING     = "#d97706"
    DANGER      = "#dc2626"


# ======================= Grad-CAM =======================
def get_gradcam_heatmap(model, img_array, hand_features=None,
                        pred_index=None,
                        layer_name='convnext_tiny_stage_3_block_2_pointwise_conv_2'):
    grad_model = None
    try:
        grad_model = tf.keras.models.Model(
            model.inputs,
            [model.get_layer(layer_name).output, model.output]
        )
    except Exception:
        for layer in reversed(model.layers):
            if 'conv' in layer.name and 'pointwise' in layer.name:
                grad_model = tf.keras.models.Model(
                    model.inputs, [layer.output, model.output]
                )
                break

    if grad_model is None:
        raise ValueError(f"Could not find layer '{layer_name}' or an alternative.")

    inputs = [img_array]
    if hand_features is not None:
        inputs.append(hand_features)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


# ======================= Reusable Widgets =======================
class Card(QFrame):
    """Clean white card with subtle border and shadow"""
    def __init__(self, title="", icon="", parent=None):
        super().__init__(parent)
        self.setObjectName("Card")
        self.setStyleSheet(f"""
            QFrame#Card {{
                background-color: {Theme.BG_CARD};
                border: 1px solid {Theme.BORDER};
                border-radius: 8px;
            }}
        """)
        shadow = QGraphicsDropShadowEffect()
        shadow.setBlurRadius(16)
        shadow.setColor(QColor(0, 0, 0, 18))
        shadow.setOffset(0, 2)
        self.setGraphicsEffect(shadow)

        self.main_layout = QVBoxLayout(self)
        self.main_layout.setContentsMargins(18, 14, 18, 18)
        self.main_layout.setSpacing(12)

        if title:
            header = QHBoxLayout()
            header.setSpacing(8)
            if icon:
                icon_lbl = QLabel(icon)
                icon_lbl.setStyleSheet("font-size: 17px;")
                header.addWidget(icon_lbl)
            title_lbl = QLabel(title)
            title_lbl.setStyleSheet(f"""
                color: {Theme.TEXT};
                font-size: 15px;
                font-weight: 600;
                letter-spacing: 0.2px;
            """)
            header.addWidget(title_lbl)
            header.addStretch()
            self.main_layout.addLayout(header)

            divider = QFrame()
            divider.setFixedHeight(1)
            divider.setStyleSheet(f"background-color: {Theme.BORDER};")
            self.main_layout.addWidget(divider)


class ImageCard(QFrame):
    """Image display card with caption"""
    def __init__(self, title, icon=""):
        super().__init__()
        self.setObjectName("ImageCard")
        self.setStyleSheet(f"""
            QFrame#ImageCard {{
                background-color: {Theme.BG_CARD};
                border: 1px solid {Theme.BORDER};
                border-radius: 8px;
            }}
        """)
        shadow = QGraphicsDropShadowEffect()
        shadow.setBlurRadius(12)
        shadow.setColor(QColor(0, 0, 0, 14))
        shadow.setOffset(0, 2)
        self.setGraphicsEffect(shadow)

        layout = QVBoxLayout(self)
        layout.setContentsMargins(12, 10, 12, 12)
        layout.setSpacing(8)

        header = QHBoxLayout()
        if icon:
            il = QLabel(icon)
            il.setStyleSheet("font-size: 15px;")
            header.addWidget(il)
        tl = QLabel(title)
        tl.setStyleSheet(f"""
            color: {Theme.TEXT_DIM};
            font-size: 13px;
            font-weight: 600;
            letter-spacing: 0.3px;
        """)
        header.addWidget(tl)
        header.addStretch()
        layout.addLayout(header)

        self.image_label = QLabel("—")
        self.image_label.setAlignment(Qt.AlignCenter)
        self.image_label.setMinimumSize(240, 240)
        self.image_label.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.image_label.setStyleSheet(f"""
            QLabel {{
                background-color: {Theme.BG_SUBTLE};
                border: 1px solid {Theme.BORDER};
                border-radius: 6px;
                color: {Theme.TEXT_DIM};
                font-size: 13px;
            }}
        """)
        layout.addWidget(self.image_label, 1)

    def set_image(self, img_rgb):
        h, w, ch = img_rgb.shape
        bytes_per_line = ch * w
        qimg = QImage(img_rgb.data, w, h, bytes_per_line, QImage.Format_RGB888)
        pix = QPixmap.fromImage(qimg)
        self.image_label.setPixmap(
            pix.scaled(self.image_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation)
        )

    def clear_image(self, placeholder="—"):
        self.image_label.setPixmap(QPixmap())
        self.image_label.setText(placeholder)


class StatusBadge(QLabel):
    def __init__(self, text="", color=Theme.TEXT_DIM):
        super().__init__(text)
        self.setAlignment(Qt.AlignCenter)
        self.setFixedHeight(28)
        self.set_color(color)

    def set_color(self, color):
        self.setStyleSheet(f"""
            QLabel {{
                background-color: {color}18;
                color: {color};
                border: 1px solid {color}40;
                border-radius: 14px;
                padding: 0 14px;
                font-weight: 600;
                font-size: 12px;
            }}
        """)

    def set_status(self, text, color):
        self.setText(text)
        self.set_color(color)


class ProbRow(QWidget):
    """Single probability row with progress bar"""
    def __init__(self, name, full_name, pct, is_top=False):
        super().__init__()
        layout = QHBoxLayout(self)
        layout.setContentsMargins(6, 6, 6, 6)
        layout.setSpacing(12)

        name_box = QVBoxLayout()
        name_box.setSpacing(1)
        lbl = QLabel(name.upper() if is_top else name)
        lbl.setStyleSheet(f"""
            color: {Theme.ACCENT if is_top else Theme.TEXT};
            font-size: 13px;
            font-weight: {'700' if is_top else '600'};
        """)
        name_box.addWidget(lbl)
        if full_name and full_name != name:
            sub = QLabel(full_name)
            sub.setStyleSheet(f"color: {Theme.TEXT_DIM}; font-size: 11px;")
            name_box.addWidget(sub)
        layout.addLayout(name_box, 3)

        bar = QProgressBar()
        bar.setRange(0, 100)
        bar.setValue(int(pct))
        bar.setTextVisible(False)
        bar.setFixedHeight(6)
        chunk_color = Theme.SUCCESS if is_top else Theme.ACCENT
        bar.setStyleSheet(f"""
            QProgressBar {{
                background-color: {Theme.BG_SUBTLE};
                border: none;
                border-radius: 3px;
            }}
            QProgressBar::chunk {{
                background-color: {chunk_color};
                border-radius: 3px;
            }}
        """)
        layout.addWidget(bar, 4)

        pct_lbl = QLabel(f"{pct:.2f}%")
        pct_lbl.setAlignment(Qt.AlignRight | Qt.AlignVCenter)
        pct_lbl.setFixedWidth(74)
        pct_lbl.setStyleSheet(f"""
            color: {Theme.SUCCESS if is_top else Theme.TEXT};
            font-size: 13px;
            font-weight: {'700' if is_top else '500'};
        """)
        layout.addWidget(pct_lbl)


# ======================= Main Application =======================
class SkinLesionApp(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Skin Lesion Classification with Grad-CAM")
        self.resize(1450, 920)
        self.setMinimumSize(1250, 820)

        self.model              = None
        self.current_image_path = ""
        self.current_img_rgb    = None
        self.current_img_bgr    = None

        self._setup_style()
        self._build_ui()

    def _setup_style(self):
        self.setStyleSheet(f"""
            QWidget {{
                background-color: {Theme.BG};
                color: {Theme.TEXT};
                font-family: 'Segoe UI', 'Arial', sans-serif;
                font-size: 14px;
            }}
            QPushButton {{
                background-color: {Theme.BG};
                color: {Theme.TEXT};
                border: 1px solid {Theme.BORDER_DARK};
                border-radius: 6px;
                padding: 10px 18px;
                font-weight: 600;
                font-size: 13px;
            }}
            QPushButton:hover {{
                background-color: {Theme.BG_SUBTLE};
                border-color: {Theme.ACCENT};
            }}
            QPushButton:pressed {{ background-color: #eef1f5; }}
            QPushButton:disabled {{
                background-color: {Theme.BG_SUBTLE};
                color: {Theme.TEXT_DIM};
                border-color: {Theme.BORDER};
            }}
            QPushButton#primaryBtn {{
                background-color: {Theme.ACCENT};
                color: white;
                border: 1px solid {Theme.ACCENT};
            }}
            QPushButton#primaryBtn:hover {{
                background-color: {Theme.ACCENT_LIGHT};
                border-color: {Theme.ACCENT_LIGHT};
            }}
            QPushButton#primaryBtn:disabled {{
                background-color: #93b4e8;
                border-color: #93b4e8;
                color: #ffffff;
            }}
            QProgressBar {{
                background-color: {Theme.BG_SUBTLE};
                border: none;
                border-radius: 3px;
                color: transparent;
            }}
            QScrollArea {{ background: transparent; border: none; }}
            QScrollBar:vertical {{
                background: transparent;
                width: 8px;
                margin: 0;
            }}
            QScrollBar::handle:vertical {{
                background: {Theme.BORDER_DARK};
                border-radius: 4px;
                min-height: 30px;
            }}
            QScrollBar::handle:vertical:hover {{ background: {Theme.ACCENT}; }}
            QScrollBar::add-line:vertical, QScrollBar::sub-line:vertical {{ height: 0; }}
        """)

    # ---------- Build UI ----------
    def _build_ui(self):
        root = QVBoxLayout(self)
        root.setContentsMargins(0, 0, 0, 0)
        root.setSpacing(0)

        root.addWidget(self._build_header())

        body_wrapper = QWidget()
        body_wrapper.setStyleSheet(f"background-color: {Theme.BG};")
        body_layout = QHBoxLayout(body_wrapper)
        body_layout.setContentsMargins(20, 18, 20, 18)
        body_layout.setSpacing(16)

        # Left: images column
        images_col = QVBoxLayout()
        images_col.setSpacing(12)

        self.card_original = ImageCard("Original Image", "🖼")
        self.card_heatmap  = ImageCard("Grad-CAM Heatmap", "🔥")
        self.card_overlay  = ImageCard("Grad-CAM Overlay", "🎯")

        images_col.addWidget(self.card_original, 1)
        images_col.addWidget(self.card_heatmap, 1)
        images_col.addWidget(self.card_overlay, 1)

        body_layout.addLayout(images_col, 5)

        # Right: results column
        results_col = QVBoxLayout()
        results_col.setSpacing(12)

        results_col.addWidget(self._build_top_result_card())

        probs_card = Card("All Class Probabilities", "📊")

        scroll = QScrollArea()
        scroll.setWidgetResizable(True)
        scroll.setStyleSheet("QScrollArea { background: transparent; border: none; }")

        self.probs_container = QWidget()
        self.probs_container.setStyleSheet("background: transparent;")
        self.probs_layout = QVBoxLayout(self.probs_container)
        self.probs_layout.setContentsMargins(0, 0, 0, 0)
        self.probs_layout.setSpacing(2)
        self.probs_layout.addStretch()
        scroll.setWidget(self.probs_container)

        probs_card.main_layout.addWidget(scroll, 1)
        results_col.addWidget(probs_card, 1)

        body_layout.addLayout(results_col, 4)

        root.addWidget(body_wrapper, 1)

        root.addWidget(self._build_bottombar())

    # ---------- Header ----------
    def _build_header(self):
        header = QFrame()
        header.setFixedHeight(82)
        header.setStyleSheet(f"""
            QFrame {{
                background-color: {Theme.BG};
                border-bottom: 1px solid {Theme.BORDER};
            }}
        """)
        layout = QHBoxLayout(header)
        layout.setContentsMargins(24, 0, 24, 0)
        layout.setSpacing(16)

        logo_box = QVBoxLayout()
        logo_box.setSpacing(3)

        title = QLabel("Fusing ConvNeXt-Tiny Deep Handcrafted")
        title.setStyleSheet(f"""
            color: {Theme.TEXT};
            font-size: 24px;
            font-weight: 700;
            letter-spacing: -0.3px;
        """)
        logo_box.addWidget(title)

        subtitle = QLabel("Deep Handcrafted Skin Lesion Classification with Grad-CAM")
        subtitle.setStyleSheet(f"color: {Theme.TEXT_DIM}; font-size: 12px;")
        logo_box.addWidget(subtitle)

        layout.addLayout(logo_box)
        layout.addStretch()

        self.model_status = StatusBadge("● No Model Loaded", Theme.WARNING)
        layout.addWidget(self.model_status)

        layout.addSpacing(8)

        self.load_model_button = QPushButton("Load Model Weights")
        self.load_model_button.clicked.connect(self.browse_model)
        layout.addWidget(self.load_model_button)

        self.browse_button = QPushButton("Open Image")
        self.browse_button.clicked.connect(self.browse_image)
        layout.addWidget(self.browse_button)

        self.predict_button = QPushButton("Run Prediction")
        self.predict_button.setObjectName("primaryBtn")
        self.predict_button.clicked.connect(self.run_model)
        self.predict_button.setEnabled(False)
        layout.addWidget(self.predict_button)

        # ✅ زر المسح الجديد
        self.clear_button = QPushButton("Clear")
        self.clear_button.clicked.connect(self.clear_all)
        layout.addWidget(self.clear_button)

        return header

    # ---------- Top Result Card ----------
    def _build_top_result_card(self):
        card = QFrame()
        card.setObjectName("TopResultCard")
        card.setStyleSheet(f"""
            QFrame#TopResultCard {{
                background-color: {Theme.SUCCESS_SOFT};
                border: 1px solid {Theme.SUCCESS}60;
                border-left: 4px solid {Theme.SUCCESS};
                border-radius: 8px;
            }}
        """)
        shadow = QGraphicsDropShadowEffect()
        shadow.setBlurRadius(16)
        shadow.setColor(QColor(5, 150, 105, 25))
        shadow.setOffset(0, 2)
        card.setGraphicsEffect(shadow)

        layout = QHBoxLayout(card)
        layout.setContentsMargins(22, 18, 22, 18)
        layout.setSpacing(16)

        text_box = QVBoxLayout()
        text_box.setSpacing(4)

        header_lbl = QLabel("TOP PREDICTION")
        header_lbl.setStyleSheet(f"""
            color: {Theme.SUCCESS};
            font-size: 12px;
            font-weight: 700;
            letter-spacing: 1.2px;
        """)
        text_box.addWidget(header_lbl)

        self.top_name_label = QLabel("—")
        self.top_name_label.setStyleSheet(f"""
            color: {Theme.TEXT};
            font-size: 24px;
            font-weight: 700;
        """)
        text_box.addWidget(self.top_name_label)

        self.top_full_label = QLabel("")
        self.top_full_label.setStyleSheet(f"color: {Theme.TEXT_DIM}; font-size: 13px;")
        text_box.addWidget(self.top_full_label)

        layout.addLayout(text_box, 1)

        self.top_conf_label = QLabel("")
        self.top_conf_label.setAlignment(Qt.AlignRight | Qt.AlignVCenter)
        self.top_conf_label.setStyleSheet(f"""
            color: {Theme.SUCCESS};
            font-size: 38px;
            font-weight: 700;
        """)
        layout.addWidget(self.top_conf_label)

        return card

    # ---------- Bottom Status Bar ----------
    def _build_bottombar(self):
        bar = QFrame()
        bar.setFixedHeight(42)
        bar.setStyleSheet(f"""
            QFrame {{
                background-color: {Theme.BG_SUBTLE};
                border-top: 1px solid {Theme.BORDER};
            }}
        """)
        layout = QHBoxLayout(bar)
        layout.setContentsMargins(24, 0, 24, 0)

        self.bottom_status = QLabel("Ready — load model weights to begin")
        self.bottom_status.setStyleSheet(f"color: {Theme.TEXT_DIM}; font-size: 12px;")
        layout.addWidget(self.bottom_status)

        layout.addStretch()

        self.time_label = QLabel("")
        self.time_label.setStyleSheet(f"color: {Theme.TEXT_DIM}; font-size: 12px;")
        layout.addWidget(self.time_label)

        return bar

    # ---------- Model loading ----------
    def browse_model(self):
        path, _ = QFileDialog.getOpenFileName(
            self, "Select Model Weights", "",
            "Keras Weights (*.h5 *.keras *.weights.h5);;All Files (*)"
        )
        if not path:
            return
        try:
            self.model_status.set_status("● Loading...", Theme.WARNING)
            self.bottom_status.setText(f"Loading: {os.path.basename(path)}")
            self.load_model_button.setEnabled(False)
            QApplication.processEvents()

            self.model = load_model_from_weights(path)
            self.model_status.set_status("● Model Ready", Theme.SUCCESS)
            self.bottom_status.setText(f"Model loaded: {os.path.basename(path)}")
            self.load_model_button.setEnabled(True)
            self._update_predict_btn()

            if self.current_img_rgb is not None:
                self.run_model()
        except Exception as e:
            self.model_status.set_status("● Load Failed", Theme.DANGER)
            self.bottom_status.setText("Model loading failed")
            self.load_model_button.setEnabled(True)
            QMessageBox.critical(self, "Model Loading Error", str(e))

    # ---------- Image loading ----------
    def browse_image(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Skin Image", "",
            "Images (*.png *.jpg *.jpeg *.bmp)"
        )
        if file_path:
            self.current_image_path = file_path
            self.load_image(file_path)

    def load_image(self, file_path):
        img_bgr = cv2.imread(file_path)
        if img_bgr is None:
            QMessageBox.warning(self, "Error", "Unable to read the image.")
            return
        self.current_img_bgr = img_bgr
        self.current_img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        self.card_original.set_image(self.current_img_rgb)
        self.card_heatmap.clear_image("Awaiting prediction")
        self.card_overlay.clear_image("Awaiting prediction")
        self.bottom_status.setText(f"Loaded: {os.path.basename(file_path)}")
        self._update_predict_btn()

        if self.model is not None:
            self.run_model()

    # ---------- Prediction ----------
    def run_model(self):
        if self.model is None:
            QMessageBox.warning(self, "No Model", "Please load model weights first.")
            return
        if self.current_img_rgb is None:
            QMessageBox.warning(self, "No Image", "Please load an image first.")
            return

        self.predict_button.setEnabled(False)
        self.bottom_status.setText("Running inference...")
        QApplication.processEvents()

        try:
            start_time = time.time()

            probs = predict_image(self.model, self.current_img_rgb)

            img_resized = cv2.resize(self.current_img_rgb, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
            img_pre     = ef_preprocess(img_resized)[np.newaxis, ...]
            hand_feat   = extract_handcrafted_features_bgr(self.current_img_bgr)[np.newaxis, ...]
            heatmap     = get_gradcam_heatmap(
                self.model, img_pre, hand_feat,
                pred_index=int(np.argmax(probs))
            )

            duration = time.time() - start_time

            self._update_results(probs, duration)

            h, w, ch = self.current_img_rgb.shape
            heatmap_resized = cv2.resize(heatmap, (w, h))
            heatmap_color = cv2.applyColorMap(
                np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
            )
            heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
            overlay = cv2.addWeighted(self.current_img_rgb, 0.6, heatmap_rgb, 0.4, 0)

            self.card_original.set_image(self.current_img_rgb)
            self.card_heatmap.set_image(heatmap_rgb)
            self.card_overlay.set_image(overlay)

            top_idx = int(np.argmax(probs))
            top_name = CLASS_NAMES[top_idx]
            self.bottom_status.setText(
                f"Prediction: {get_full_name(top_name)} ({probs[top_idx]*100:.2f}%)"
            )
            self.time_label.setText(f"Inference: {duration:.2f}s")
        except Exception as e:
            QMessageBox.critical(self, "Prediction Error", str(e))
            self.bottom_status.setText("Prediction failed")
        finally:
            self.predict_button.setEnabled(True)

    def _update_results(self, probs, duration):
        top_idx = int(np.argmax(probs))
        top_name = CLASS_NAMES[top_idx]
        top_full = get_full_name(top_name)

        self.top_name_label.setText(top_name.upper())
        self.top_full_label.setText(top_full)
        self.top_conf_label.setText(f"{probs[top_idx]*100:.2f}%")

        while self.probs_layout.count() > 1:
            item = self.probs_layout.takeAt(0)
            if item.widget():
                item.widget().deleteLater()

        sorted_idx = np.argsort(probs)[::-1]
        for rank, i in enumerate(sorted_idx):
            name = CLASS_NAMES[i]
            full = get_full_name(name)
            pct = float(probs[i] * 100)
            row = ProbRow(name, full, pct, is_top=(i == top_idx))
            self.probs_layout.insertWidget(rank, row)

    def _update_predict_btn(self):
        self.predict_button.setEnabled(
            self.model is not None and self.current_img_rgb is not None
        )

    # ---------- ✅ Clear All (جديد) ----------
    def clear_all(self):
        """مسح الصورة والنتائج وإعادة الواجهة إلى حالتها الأولية"""
        # مسح الحالة الداخلية
        self.current_image_path = ""
        self.current_img_rgb    = None
        self.current_img_bgr    = None

        # مسح بطاقات الصور
        self.card_original.clear_image("—")
        self.card_heatmap.clear_image("—")
        self.card_overlay.clear_image("—")

        # مسح بطاقة أعلى نتيجة
        self.top_name_label.setText("—")
        self.top_full_label.setText("")
        self.top_conf_label.setText("")

        # مسح قائمة الاحتمالات
        while self.probs_layout.count() > 1:
            item = self.probs_layout.takeAt(0)
            if item.widget():
                item.widget().deleteLater()

        # مسح شريط الحالة
        self.bottom_status.setText("Cleared — load an image to begin")
        self.time_label.setText("")

        # تعطيل زر التنبؤ
        self._update_predict_btn()


# ======================= main =======================
if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    window = SkinLesionApp()
    window.show()
    sys.exit(app.exec_())

C:\Users\Lenovo\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\Lenovo\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\Lenovo\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


C:\Users\Lenovo\anaconda3\Lib\site-packages\skimage\feature\texture.py:360: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
